In [1]:
import sys
import os

# 1. Direct Python to the root project folder
sys.path.append(os.path.abspath(os.path.join('..')))

In [6]:
from scripts.ingest import load_faq_data
documents = load_faq_data()

Existing Ecommerce FAQ With Ids loaded Successfully!!
Document Size:79


In [4]:
len(documents)

79

In [8]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

qDNpG85o
How can I create an account?
To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.


In [9]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [10]:
data_gen_instructions = """
You emulate a ecommerce customer who is asking questions.
Formulate 5 questions this customer might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [11]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [12]:
import json
user_prompt = json.dumps(doc)

In [13]:
user_prompt

'{"question": "How can I create an account?", "answer": "To create an account, click on the \'Sign Up\' button on the top right corner of our website and follow the instructions to complete the registration process.", "id": "qDNpG85o"}'

In [14]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [15]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [16]:
result = response.output_parsed

print(result)

questions=['How do I sign up for an account on your website?', 'Where do I click to create a new account?', 'What’s the easiest way to register on your site?', 'Can you tell me how to make an account from the homepage?', 'Which button do I use to start creating an account?']


In [17]:
from scripts.evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['How do I sign up for an account on your website?', 'Where can I find the Sign Up button to register?', 'What steps do I need to follow to create a new account?', 'Can I make an account from the top right corner of the site?', 'How do I complete the registration process for a new account?']


In [20]:
usage

ResponseUsage(input_tokens=188, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=78, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=266)

In [19]:
from scripts.evaluation_utils import calc_price
calc_price(usage)

{'input_cost': 0.00014099999999999998,
 'output_cost': 0.00035099999999999997,
 'total_cost': 0.0004919999999999999}

In [21]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'How do I sign up for an account on your website?',
  'document': 'qDNpG85o'},
 {'question': 'Where can I find the Sign Up button to register?',
  'document': 'qDNpG85o'},
 {'question': 'What steps do I need to follow to create a new account?',
  'document': 'qDNpG85o'},
 {'question': 'Can I make an account from the top right corner of the site?',
  'document': 'qDNpG85o'},
 {'question': 'How do I complete the registration process for a new account?',
  'document': 'qDNpG85o'}]

In [22]:
import pandas as pd

pd.DataFrame(records)

,question,document
0,How do I sign up for an account on your website?,qDNpG85o
1,Where can I find the Sign Up button to register?,qDNpG85o
2,What steps do I need to follow to create a new...,qDNpG85o
3,Can I make an account from the top right corne...,qDNpG85o
4,How do I complete the registration process for...,qDNpG85o


In [23]:
from scripts.evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [24]:
generate_ground_truth(doc)

([{'question': 'How do I sign up for an account on your website?',
   'document': 'qDNpG85o'},
  {'question': 'Where can I find the Sign Up button to create an account?',
   'document': 'qDNpG85o'},
  {'question': 'What steps do I need to follow to register a new account?',
   'document': 'qDNpG85o'},
  {'question': 'How do I make an account on the top right of the site?',
   'document': 'qDNpG85o'},
  {'question': 'Can you tell me how to complete the account registration process?',
   'document': 'qDNpG85o'}],
 ResponseUsage(input_tokens=188, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=80, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=268))

In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/79 [00:00<?, ?it/s]

In [25]:
from concurrent.futures import ThreadPoolExecutor
from scripts.evaluation_utils import map_progress

In [26]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/79 [00:00<?, ?it/s]

In [27]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

395

In [28]:
from scripts.evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.04447049999999999

In [29]:
df_ground_truth = pd.DataFrame(ground_truth)

In [30]:
df_ground_truth

,question,document
0,How do I make an account on your website?,qDNpG85o
1,Where do I click to sign up for a new account?,qDNpG85o
2,What’s the easiest way to register on your site?,qDNpG85o
3,Can you tell me how to create an account online?,qDNpG85o
4,How do I complete the account registration pro...,qDNpG85o
...,...,...
390,Can I return something I bought during a sale ...,cSa61muY
391,If I purchased an item during a promotional ev...,cSa61muY
392,"When returning a product bought on promotion, ...",cSa61muY
393,Are items bought with a promo discount still r...,cSa61muY


In [31]:
df_ground_truth.to_csv("../data/ground_truth.csv", index=False)